In [1]:
train_dir = '/content/drive/MyDrive/Colab Notebooks/data'
test_dir = '/content/drive/MyDrive/Colab Notebooks/NewArts2'
aug_dir='/content/drive/MyDrive/Colab Notebooks/Augmented'


In [2]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torchvision.transforms import v2
from torch.utils.data import DataLoader, random_split
from PIL import Image
import cv2
from sklearn.metrics import classification_report, f1_score, accuracy_score
import numpy as np

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import os
classes = ["happy", "sad", "anger", "fear"]

for class_name in classes:

    input_folder = os.path.join(train_dir, class_name)
    output_folder = os.path.join(aug_dir, class_name)

    os.makedirs(output_folder, exist_ok=True)

    print("Processing:", class_name)


Processing: happy
Processing: sad
Processing: anger
Processing: fear


In [5]:
import shutil
import os

aug_dir = '/content/drive/MyDrive/Colb Notebooks/Augmented' # wait, check your aug_dir path spelling!
# Make sure it matches your actual path:
aug_dir = '/content/drive/MyDrive/Colab Notebooks/Augmented'

if os.path.exists(aug_dir):
    shutil.rmtree(aug_dir)
    print("🧹 Cleared old augmented folders!")

🧹 Cleared old augmented folders!


In [6]:
classes = [
    item for item in os.listdir(train_dir)
    if os.path.isdir(os.path.join(train_dir, item))
]

print("Classes:", classes)

for class_name in classes:
    class_folder = os.path.join(train_dir, class_name)

    images = [
        f for f in os.listdir(class_folder)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ]

    print(class_name, ":", len(images), "images")

Classes: ['Happy', 'Sad', 'Fear', 'Angry']
Happy : 171 images
Sad : 183 images
Fear : 179 images
Angry : 169 images


In [7]:
import os
import random

random.seed(42)

train_files = []
val_files = []

for class_name in classes:

    class_folder = os.path.join(train_dir, class_name)

    images = [
        f for f in os.listdir(class_folder)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ]

    random.shuffle(images)

    split = int(0.8 * len(images))

    class_train = images[:split]
    class_val = images[split:]

    for img in class_train:
        train_files.append((class_name, img))

    for img in class_val:
        val_files.append((class_name, img))

print("\n========== ORIGINAL SPLIT ==========")

print("Training   :", len(train_files))
print("Validation :", len(val_files))

print("\nPer class:")

for class_name in classes:

    train_count = sum(
        1 for c, img in train_files
        if c == class_name
    )

    val_count = sum(
        1 for c, img in val_files
        if c == class_name
    )

    print(
        f"{class_name}: "
        f"{train_count} train | "
        f"{val_count} validation"
    )


========== ORIGINAL SPLIT ==========
Training   : 560
Validation : 142

Per class:
Happy: 136 train | 35 validation
Sad: 146 train | 37 validation
Fear: 143 train | 36 validation
Angry: 135 train | 34 validation


In [8]:
# ============================================================
# CREATE AUGMENTED IMAGES FROM TRAINING IMAGES ONLY
# ============================================================

import os
import shutil
from PIL import Image
from torchvision import transforms

# ------------------------------------------------------------
# Remove old Augmented folder
# ------------------------------------------------------------

if os.path.exists(aug_dir):
    shutil.rmtree(aug_dir)

os.makedirs(aug_dir)

# Create class folders
for class_name in classes:
    os.makedirs(os.path.join(aug_dir, class_name))


# ------------------------------------------------------------
# Augmentation
# ------------------------------------------------------------

augment = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.CenterCrop(224),
    transforms.RandomGrayscale(p=0.2)



])

num_copies = 6

augmented_count = 0


# ------------------------------------------------------------
# Augment ONLY train_files
# ------------------------------------------------------------

for class_name, img_name in train_files:

    input_path = os.path.join(
        train_dir,
        class_name,
        img_name
    )

    output_folder = os.path.join(
        aug_dir,
        class_name
    )

    try:

        image = Image.open(input_path).convert('RGB')

        name = os.path.splitext(img_name)[0]

        for i in range(num_copies):

            new_image = augment(image)

            save_path = os.path.join(
                output_folder,
                f'{name}_aug_{i}.jpg'
            )

            new_image.save(save_path, quality=95)

            augmented_count += 1

    except Exception as e:
        print("Error:", input_path, e)


# ------------------------------------------------------------
# RESULT
# ------------------------------------------------------------

print("\n" + "=" * 50)
print("       AUGMENTATION COMPLETE")
print("=" * 50)

print("Original training images :", len(train_files))
print("Validation images        :", len(val_files))
print("Augmented images         :", augmented_count)

print("-" * 50)

print("Expected augmented       :", len(train_files) * 4)
print("Final training images    :", len(train_files) + augmented_count)
print("Validation images        :", len(val_files))
print("Test images              :", 401)

print("=" * 50)


       AUGMENTATION COMPLETE
Original training images : 560
Validation images        : 142
Augmented images         : 3360
--------------------------------------------------
Expected augmented       : 2240
Final training images    : 3920
Validation images        : 142
Test images              : 401


In [9]:
# ============================================================
# LOAD TRAIN / VALIDATION / TEST DATA
# ============================================================

import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, ConcatDataset, Subset

# ------------------------------------------------------------
# Transforms
# ------------------------------------------------------------

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# ============================================================
# ORIGINAL DATASET
# ============================================================

original_dataset = datasets.ImageFolder(
    root=train_dir,
    transform=train_transform
)


# ============================================================
# GET THE SAME 560 TRAINING IMAGES
# ============================================================

train_indices = []

for class_name, img_name in train_files:

    image_path = os.path.join(
        train_dir,
        class_name,
        img_name
    )

    for idx, (path, label) in enumerate(original_dataset.samples):

        if path == image_path:
            train_indices.append(idx)
            break


original_train_dataset = Subset(
    original_dataset,
    train_indices
)


# ============================================================
# AUGMENTED DATASET
# ============================================================

aug_dataset = datasets.ImageFolder(
    root=aug_dir,
    transform=train_transform
)


# ============================================================
# COMBINE ORIGINAL + AUGMENTED
# ============================================================

train_dataset = ConcatDataset([
    original_train_dataset,
    aug_dataset
])


# ============================================================
# VALIDATION
# ============================================================

val_full_dataset = datasets.ImageFolder(
    root=train_dir,
    transform=val_test_transform
)

val_indices = []

for class_name, img_name in val_files:

    image_path = os.path.join(
        train_dir,
        class_name,
        img_name
    )

    for idx, (path, label) in enumerate(val_full_dataset.samples):

        if path == image_path:
            val_indices.append(idx)
            break


val_dataset = Subset(
    val_full_dataset,
    val_indices
)


# ============================================================
# TEST
# ============================================================

test_dataset = datasets.ImageFolder(
    root=test_dir,
    transform=val_test_transform
)


# ============================================================
# DATA LOADERS
# ============================================================

batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2
)


# ============================================================
# FINAL CHECK
# ============================================================

print("=" * 50)
print("             FINAL DATA")
print("=" * 50)

print("Original training :", len(original_train_dataset))
print("Augmented training :", len(aug_dataset))
print("TOTAL TRAINING     :", len(train_dataset))
print("VALIDATION         :", len(val_dataset))
print("TEST               :", len(test_dataset))

print("=" * 50)

print("Classes:", original_dataset.classes)

             FINAL DATA
Original training : 560
Augmented training : 3360
TOTAL TRAINING     : 3920
VALIDATION         : 142
TEST               : 407
Classes: ['Angry', 'Fear', 'Happy', 'Sad']


In [ ]:
import torch.optim as optim

In [ ]:
from torchvision import models

In [ ]:
pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 26.0 MB/s eta 0:00:00


In [10]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

In [ ]:
# ============================================================
# EFFICIENTNET-B0
# Children's Drawing Emotion Analysis
# ============================================================

import os
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import models
from torchvision.models import EfficientNet_B0_Weights

from sklearn.metrics import accuracy_score, f1_score, classification_report


# ============================================================
# 1. SETTINGS
# ============================================================

NUM_CLASSES = 4

EPOCHS = 30
BATCH_SIZE = 32

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

PATIENCE = 10

SAVE_DIR = (
    "/content/drive/MyDrive/Colab Notebooks/"
    "EfficientNetB0_Simple"
)

os.makedirs(SAVE_DIR, exist_ok=True)

print("Using device:", device)


# ============================================================
# 2. DATALOADERS
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


# ============================================================
# 3. CREATE EFFICIENTNET-B0
# ============================================================

model = models.efficientnet_b0(
    weights=EfficientNet_B0_Weights.DEFAULT
)


# ============================================================
# 4. FREEZE THE FEATURE EXTRACTOR
# ============================================================

for parameter in model.features.parameters():
    parameter.requires_grad = False


# ============================================================
# 5. REPLACE THE CLASSIFIER
# ============================================================

num_features = model.classifier[1].in_features

model.classifier = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(num_features, NUM_CLASSES)
)

model = model.to(device)


# ============================================================
# 6. LOSS FUNCTION
# ============================================================

criterion = nn.CrossEntropyLoss(
    label_smoothing=0.1
)


# ============================================================
# 7. OPTIMIZER
# ============================================================

optimizer = optim.AdamW(
    model.classifier.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)


# ============================================================
# 8. TRAINING
# ============================================================

best_f1 = 0
best_epoch = 0
no_improvement = 0

history = []


for epoch in range(EPOCHS):

    # --------------------------------------------------------
    # TRAIN
    # --------------------------------------------------------

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        running_loss += (
            loss.item() * images.size(0)
        )

        predictions = outputs.argmax(
            dim=1
        )

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total


    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------

    model.eval()

    val_loss = 0

    all_labels = []
    all_predictions = []

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            val_loss += (
                loss.item() * images.size(0)
            )

            predictions = outputs.argmax(
                dim=1
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

    val_loss = val_loss / len(val_dataset)

    val_acc = accuracy_score(
        all_labels,
        all_predictions
    )

    val_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )


    # --------------------------------------------------------
    # PRINT RESULTS
    # --------------------------------------------------------

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc*100:.2f}% | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc*100:.2f}% | "
        f"Val F1: {val_f1:.4f}"
    )


    # --------------------------------------------------------
    # SAVE HISTORY
    # --------------------------------------------------------

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "val_loss": val_loss,
        "val_accuracy": val_acc,
        "val_macro_f1": val_f1
    })


    # --------------------------------------------------------
    # SAVE EVERY EPOCH
    # --------------------------------------------------------

    torch.save(
        {
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_accuracy": val_acc,
            "val_macro_f1": val_f1
        },
        os.path.join(
            SAVE_DIR,
            f"epoch_{epoch+1}.pth"
        )
    )


    # --------------------------------------------------------
    # SAVE BEST MODEL
    # --------------------------------------------------------

    if val_f1 > best_f1:

        best_f1 = val_f1
        best_epoch = epoch + 1
        no_improvement = 0

        torch.save(
            model.state_dict(),
            os.path.join(
                SAVE_DIR,
                "best_efficientnet_b0.pth"
            )
        )

        print(
            "  ✓ Best model saved!"
        )

    else:

        no_improvement += 1

        print(
            f"  No improvement "
            f"({no_improvement}/{PATIENCE})"
        )


    # --------------------------------------------------------
    # EARLY STOPPING
    # --------------------------------------------------------

    if no_improvement >= PATIENCE:

        print("Early stopping!")

        break


# ============================================================
# 9. LOAD BEST MODEL
# ============================================================

model.load_state_dict(
    torch.load(
        os.path.join(
            SAVE_DIR,
            "best_efficientnet_b0.pth"
        ),
        map_location=device
    )
)

print(
    f"\nBest model came from epoch {best_epoch}"
)


# ============================================================
# 10. TEST
# ============================================================

model.eval()

test_labels = []
test_predictions = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)

        outputs = model(images)

        predictions = outputs.argmax(
            dim=1
        )

        test_labels.extend(
            labels.numpy()
        )

        test_predictions.extend(
            predictions.cpu().numpy()
        )


# ============================================================
# 11. FINAL RESULTS
# ============================================================

test_accuracy = accuracy_score(
    test_labels,
    test_predictions
)

test_f1 = f1_score(
    test_labels,
    test_predictions,
    average="macro",
    zero_division=0
)

print("\n" + "=" * 60)
print("FINAL TEST RESULTS")
print("=" * 60)

print(
    f"Test Accuracy: "
    f"{test_accuracy*100:.2f}%"
)

print(
    f"Test Macro F1: "
    f"{test_f1:.4f}"
)


# ============================================================
# 12. CLASSIFICATION REPORT
# ============================================================

class_names = [
    "Angry",
    "Fear",
    "Happy",
    "Sad"
]

print("\nClassification Report:")

print(
    classification_report(
        test_labels,
        test_predictions,
        target_names=class_names,
        zero_division=0
    )
)

Using device: cuda
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 142MB/s]


Epoch [1/30] Train Loss: 1.3711 | Train Acc: 31.73% | Val Loss: 1.3345 | Val Acc: 42.25% | Val F1: 0.3858
  ✓ Best model saved!
Epoch [2/30] Train Loss: 1.2937 | Train Acc: 42.17% | Val Loss: 1.2765 | Val Acc: 44.37% | Val F1: 0.4050
  ✓ Best model saved!
Epoch [3/30] Train Loss: 1.2383 | Train Acc: 48.83% | Val Loss: 1.2374 | Val Acc: 46.48% | Val F1: 0.4343
  ✓ Best model saved!
Epoch [4/30] Train Loss: 1.1935 | Train Acc: 52.02% | Val Loss: 1.2086 | Val Acc: 47.18% | Val F1: 0.4470
  ✓ Best model saved!
Epoch [5/30] Train Loss: 1.1622 | Train Acc: 55.23% | Val Loss: 1.1895 | Val Acc: 46.48% | Val F1: 0.4397
  No improvement (1/10)
Epoch [6/30] Train Loss: 1.1400 | Train Acc: 56.45% | Val Loss: 1.1747 | Val Acc: 47.18% | Val F1: 0.4484
  ✓ Best model saved!
Epoch [7/30] Train Loss: 1.1153 | Train Acc: 58.32% | Val Loss: 1.1590 | Val Acc: 47.89% | Val F1: 0.4547
  ✓ Best model saved!
Epoch [8/30] Train Loss: 1.0923 | Train Acc: 59.80% | Val Loss: 1.1517 | Val Acc: 48.59% | Val F1: 0.4

In [ ]:
# ============================================================
# 13. DOWNLOAD BEST MODEL
# ============================================================

from google.colab import files

best_model_path = os.path.join(
    SAVE_DIR,
    "best_efficientnet_b0.pth"
)

files.download(best_model_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# MOBILENETV3-LARGE
# Children's Drawing Emotion Analysis
# ============================================================

import os
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import models
from torchvision.models import MobileNet_V3_Large_Weights

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)


# ============================================================
# 1. SETTINGS
# ============================================================

NUM_CLASSES = 4

EPOCHS = 30
BATCH_SIZE = 32

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

PATIENCE = 10

SAVE_DIR = (
    "/content/drive/MyDrive/Colab Notebooks/"
    "MobileNetV3Large_Simple"
)

os.makedirs(SAVE_DIR, exist_ok=True)

print("Using device:", device)


# ============================================================
# 2. DATALOADERS
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


# ============================================================
# 3. CREATE MOBILENETV3-LARGE
# ============================================================

model = models.mobilenet_v3_large(
    weights=MobileNet_V3_Large_Weights.DEFAULT
)


# ============================================================
# 4. FREEZE THE FEATURE EXTRACTOR
# ============================================================

for parameter in model.features.parameters():
    parameter.requires_grad = False


# ============================================================
# 5. REPLACE THE CLASSIFIER
# ============================================================

num_features = model.classifier[0].in_features

model.classifier = nn.Sequential(
    nn.Linear(
        num_features,
        1280
    ),
    nn.Hardswish(),
    nn.Dropout(0.5),
    nn.Linear(
        1280,
        NUM_CLASSES
    )
)

model = model.to(device)


# ============================================================
# 6. LOSS FUNCTION
# ============================================================

criterion = nn.CrossEntropyLoss(
    label_smoothing=0.1
)


# ============================================================
# 7. OPTIMIZER
# ============================================================

optimizer = optim.AdamW(
    model.classifier.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)


# ============================================================
# 8. TRAINING
# ============================================================

best_f1 = 0
best_epoch = 0
no_improvement = 0

history = []


for epoch in range(EPOCHS):

    # ========================================================
    # TRAIN
    # ========================================================

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        running_loss += (
            loss.item() * images.size(0)
        )

        predictions = outputs.argmax(
            dim=1
        )

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

    train_loss = running_loss / total

    train_acc = correct / total


    # ========================================================
    # VALIDATION
    # ========================================================

    model.eval()

    val_loss = 0

    all_labels = []
    all_predictions = []

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            val_loss += (
                loss.item() * images.size(0)
            )

            predictions = outputs.argmax(
                dim=1
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

    val_loss = val_loss / len(val_dataset)

    val_acc = accuracy_score(
        all_labels,
        all_predictions
    )

    val_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )


    # ========================================================
    # PRINT RESULTS
    # ========================================================

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc*100:.2f}% | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc*100:.2f}% | "
        f"Val F1: {val_f1:.4f}"
    )


    # ========================================================
    # SAVE HISTORY
    # ========================================================

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "val_loss": val_loss,
        "val_accuracy": val_acc,
        "val_macro_f1": val_f1
    })


    # ========================================================
    # SAVE EVERY EPOCH
    # ========================================================

    torch.save(
        {
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_accuracy": val_acc,
            "val_macro_f1": val_f1
        },
        os.path.join(
            SAVE_DIR,
            f"epoch_{epoch+1}.pth"
        )
    )


    # ========================================================
    # SAVE BEST MODEL
    # ========================================================

    if val_f1 > best_f1:

        best_f1 = val_f1
        best_epoch = epoch + 1
        no_improvement = 0

        torch.save(
            model.state_dict(),
            os.path.join(
                SAVE_DIR,
                "best_mobilenet_v3_large.pth"
            )
        )

        print(
            "  ✓ Best model saved!"
        )

    else:

        no_improvement += 1

        print(
            f"  No improvement "
            f"({no_improvement}/{PATIENCE})"
        )


    # ========================================================
    # EARLY STOPPING
    # ========================================================

    if no_improvement >= PATIENCE:

        print("Early stopping!")

        break


# ============================================================
# 9. LOAD BEST MODEL
# ============================================================

best_model_path = os.path.join(
    SAVE_DIR,
    "best_mobilenet_v3_large.pth"
)

model.load_state_dict(
    torch.load(
        best_model_path,
        map_location=device
    )
)

print(
    f"\nBest model came from epoch {best_epoch}"
)


# ============================================================
# 10. TEST
# ============================================================

model.eval()

test_labels = []
test_predictions = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)

        outputs = model(images)

        predictions = outputs.argmax(
            dim=1
        )

        test_labels.extend(
            labels.numpy()
        )

        test_predictions.extend(
            predictions.cpu().numpy()
        )


# ============================================================
# 11. FINAL RESULTS
# ============================================================

test_accuracy = accuracy_score(
    test_labels,
    test_predictions
)

test_f1 = f1_score(
    test_labels,
    test_predictions,
    average="macro",
    zero_division=0
)

print("\n" + "=" * 60)
print("FINAL TEST RESULTS")
print("=" * 60)

print(
    f"Test Accuracy: "
    f"{test_accuracy*100:.2f}%"
)

print(
    f"Test Macro F1: "
    f"{test_f1:.4f}"
)


# ============================================================
# 12. CLASSIFICATION REPORT
# ============================================================

class_names = [
    "Angry",
    "Fear",
    "Happy",
    "Sad"
]

print("\nClassification Report:")

print(
    classification_report(
        test_labels,
        test_predictions,
        target_names=class_names,
        zero_division=0
    )
)


# ============================================================
# 13. DOWNLOAD BEST MODEL
# ============================================================

from google.colab import files

print("\nDownloading best MobileNetV3-Large model...")

files.download(
    best_model_path
)

Using device: cuda
Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-5c1a4163.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 141MB/s]


Epoch [1/30] Train Loss: 1.2072 | Train Acc: 51.71% | Val Loss: 1.1280 | Val Acc: 49.30% | Val F1: 0.4676
  ✓ Best model saved!
Epoch [2/30] Train Loss: 1.0331 | Train Acc: 61.84% | Val Loss: 1.1020 | Val Acc: 50.70% | Val F1: 0.5042
  ✓ Best model saved!
Epoch [3/30] Train Loss: 0.9543 | Train Acc: 67.73% | Val Loss: 1.1020 | Val Acc: 51.41% | Val F1: 0.5083
  ✓ Best model saved!
Epoch [4/30] Train Loss: 0.9018 | Train Acc: 70.77% | Val Loss: 1.1146 | Val Acc: 54.23% | Val F1: 0.5328
  ✓ Best model saved!
Epoch [5/30] Train Loss: 0.8605 | Train Acc: 72.81% | Val Loss: 1.1228 | Val Acc: 47.89% | Val F1: 0.4766
  No improvement (1/10)
Epoch [6/30] Train Loss: 0.8204 | Train Acc: 76.10% | Val Loss: 1.1245 | Val Acc: 50.70% | Val F1: 0.5010
  No improvement (2/10)
Epoch [7/30] Train Loss: 0.7952 | Train Acc: 77.81% | Val Loss: 1.1392 | Val Acc: 51.41% | Val F1: 0.5091
  No improvement (3/10)
Epoch [8/30] Train Loss: 0.7711 | Train Acc: 79.21% | Val Loss: 1.1369 | Val Acc: 50.70% | Val F1:

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# 13. DOWNLOAD BEST MODEL
# ============================================================

from google.colab import files

print("\nDownloading best MobileNetV3-Large model...")

files.download(
    best_model_path
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# 11. FINAL RESULTS
# ============================================================

test_accuracy = accuracy_score(
    test_labels,
    test_predictions
)

test_f1 = f1_score(
    test_labels,
    test_predictions,
    average="macro",
    zero_division=0
)

print("\n" + "=" * 60)
print("FINAL TEST RESULTS")
print("=" * 60)

print(
    f"Test Accuracy: "
    f"{test_accuracy*100:.2f}%"
)

print(
    f"Test Macro F1: "
    f"{test_f1:.4f}"
)


# ============================================================
# 12. CLASSIFICATION REPORT
# ============================================================

class_names = [
    "Angry",
    "Fear",
    "Happy",
    "Sad"
]

print("\nClassification Report:")

print(
    classification_report(
        test_labels,
        test_predictions,
        target_names=class_names,
        zero_division=0
    )
)




FINAL TEST RESULTS
Test Accuracy: 63.88%
Test Macro F1: 0.6293

Classification Report:
              precision    recall  f1-score   support

       Angry       0.63      0.57      0.60       100
        Fear       0.59      0.51      0.55       101
       Happy       0.73      0.97      0.83       100
         Sad       0.57      0.51      0.54       106

    accuracy                           0.64       407
   macro avg       0.63      0.64      0.63       407
weighted avg       0.63      0.64      0.63       407



In [11]:
import os
import json
import copy
import random
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import models
from torchvision.models import ResNet18_Weights

from sklearn.metrics import accuracy_score, f1_score

import optuna

# ============================================================
# 1. DEVICE
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

# ============================================================
# 2. SETTINGS
# ============================================================

NUM_CLASSES = 4

NUM_EPOCHS = 15
PATIENCE = 3

RESULTS_DIR = "/content/drive/MyDrive/Colab Notebooks/ResNet18_Optuna_Results"

os.makedirs(RESULTS_DIR, exist_ok=True)

print("Results will be saved to:")
print(RESULTS_DIR)

# ============================================================
# 3. CLASS NAMES
# ============================================================

class_names = [
    "Angry",
    "Fear",
    "Happy",
    "Sad"
]

# ============================================================
# 4. CREATE RESNET18 MODEL
# ============================================================

def create_resnet18(dropout, freeze_backbone):

    # Load pretrained ResNet18
    model = models.resnet18(
        weights=ResNet18_Weights.DEFAULT
    )

    # --------------------------------------------------------
    # Freeze / unfreeze backbone
    # --------------------------------------------------------

    if freeze_backbone:

        for param in model.parameters():
            param.requires_grad = False

    # --------------------------------------------------------
    # Replace final classifier
    # --------------------------------------------------------

    num_features = model.fc.in_features

    model.fc = nn.Sequential(
        nn.Dropout(dropout),
        nn.Linear(num_features, NUM_CLASSES)
    )

    # Make sure classifier is trainable
    for param in model.fc.parameters():
        param.requires_grad = True

    return model.to(device)

# ============================================================
# 5. TRAIN ONE EPOCH
# ============================================================

def train_one_epoch(model, loader, criterion, optimizer):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        # Clear old gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        # Statistics
        running_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc

# ============================================================
# 6. VALIDATION
# ============================================================

def validate(model, loader, criterion):

    model.eval()

    running_loss = 0.0

    all_labels = []
    all_predictions = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            _, predictions = torch.max(outputs, 1)

            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predictions.cpu().numpy())

    total = len(all_labels)

    val_loss = running_loss / total

    val_acc = accuracy_score(
        all_labels,
        all_predictions
    )

    val_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )

    return val_loss, val_acc, val_f1

# ============================================================
# 7. OPTUNA OBJECTIVE
# ============================================================

def objective(trial):

    print("\n" + "=" * 70)
    print(f"STARTING OPTUNA TRIAL {trial.number}")
    print("=" * 70)

    # --------------------------------------------------------
    # Hyperparameters
    # --------------------------------------------------------

    lr = trial.suggest_float(
        "lr",
        1e-5,
        1e-3,
        log=True
    )

    batch_size = trial.suggest_categorical(
        "batch_size",
        [16, 32, 64]
    )

    weight_decay = trial.suggest_float(
        "weight_decay",
        1e-5,
        1e-3,
        log=True
    )

    dropout = trial.suggest_float(
        "dropout",
        0.1,
        0.5
    )

    freeze_backbone = trial.suggest_categorical(
        "freeze_backbone",
        [True, False]
    )

    print("Hyperparameters:")
    print("Learning rate :", lr)
    print("Batch size    :", batch_size)
    print("Weight decay  :", weight_decay)
    print("Dropout       :", dropout)
    print("Freeze model  :", freeze_backbone)

    # --------------------------------------------------------
    # IMPORTANT:
    # Create loaders with this trial's batch size
    # --------------------------------------------------------

    train_loader_trial = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )

    val_loader_trial = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    model = create_resnet18(
        dropout=dropout,
        freeze_backbone=freeze_backbone
    )

    # --------------------------------------------------------
    # Loss
    # --------------------------------------------------------

    criterion = nn.CrossEntropyLoss()

    # --------------------------------------------------------
    # Optimizer
    # --------------------------------------------------------

    optimizer = optim.AdamW(
        filter(
            lambda p: p.requires_grad,
            model.parameters()
        ),
        lr=lr,
        weight_decay=weight_decay
    )

    # --------------------------------------------------------
    # Scheduler
    # --------------------------------------------------------

    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=NUM_EPOCHS
    )

    # --------------------------------------------------------
    # Trial folder
    # --------------------------------------------------------

    trial_dir = os.path.join(
        RESULTS_DIR,
        f"trial_{trial.number}"
    )

    os.makedirs(trial_dir, exist_ok=True)

    checkpoint_dir = os.path.join(
        trial_dir,
        "checkpoints"
    )

    os.makedirs(checkpoint_dir, exist_ok=True)

    # Save hyperparameters
    hyperparameters = {
        "trial": trial.number,
        "lr": lr,
        "batch_size": batch_size,
        "weight_decay": weight_decay,
        "dropout": dropout,
        "freeze_backbone": freeze_backbone,
        "epochs": NUM_EPOCHS,
        "patience": PATIENCE
    }

    with open(
        os.path.join(
            trial_dir,
            "hyperparameters.json"
        ),
        "w"
    ) as f:

        json.dump(
            hyperparameters,
            f,
            indent=4
        )

    # --------------------------------------------------------
    # Training history
    # --------------------------------------------------------

    history = []

    best_val_acc = 0.0
    best_val_f1 = 0.0
    best_epoch = 0

    epochs_without_improvement = 0

    # --------------------------------------------------------
    # TRAINING
    # --------------------------------------------------------

    for epoch in range(NUM_EPOCHS):

        train_loss, train_acc = train_one_epoch(
            model,
            train_loader_trial,
            criterion,
            optimizer
        )

        val_loss, val_acc, val_f1 = validate(
            model,
            val_loader_trial,
            criterion
        )

        scheduler.step()

        # ----------------------------------------------------
        # Save epoch results
        # ----------------------------------------------------

        epoch_result = {
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_accuracy": train_acc,
            "val_loss": val_loss,
            "val_accuracy": val_acc,
            "val_macro_f1": val_f1,
            "learning_rate": optimizer.param_groups[0]["lr"]
        }

        history.append(epoch_result)

        print(
            f"Epoch [{epoch + 1}/{NUM_EPOCHS}] "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_acc * 100:.2f}% | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc * 100:.2f}% | "
            f"Val F1: {val_f1:.4f}"
        )

        # Save history after EVERY epoch
        with open(
            os.path.join(
                trial_dir,
                "training_history.json"
            ),
            "w"
        ) as f:

            json.dump(
                history,
                f,
                indent=4
            )

        # ----------------------------------------------------
        # BEST MODEL
        # ----------------------------------------------------

        if val_acc > best_val_acc:

            best_val_acc = val_acc
            best_val_f1 = val_f1
            best_epoch = epoch + 1

            epochs_without_improvement = 0

            torch.save(
                {
                    "epoch": epoch + 1,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "val_accuracy": val_acc,
                    "val_macro_f1": val_f1,
                    "hyperparameters": hyperparameters
                },
                os.path.join(
                    checkpoint_dir,
                    "best_model.pth"
                )
            )

            print(
                f"  ✓ Best model saved! "
                f"({val_acc * 100:.2f}%)"
            )

        else:

            epochs_without_improvement += 1

            print(
                f"  No improvement "
                f"({epochs_without_improvement}/{PATIENCE})"
            )

        # ----------------------------------------------------
        # Optuna pruning
        # ----------------------------------------------------

        trial.report(
            val_acc,
            epoch
        )

        if trial.should_prune():

            print("Trial pruned by Optuna.")

            raise optuna.TrialPruned()

        # ----------------------------------------------------
        # Early stopping
        # ----------------------------------------------------

        if epochs_without_improvement >= PATIENCE:

            print("Early stopping triggered!")

            break

    # ========================================================
    # SAVE TRIAL SUMMARY
    # ========================================================

    summary = {
        "trial": trial.number,
        "best_epoch": best_epoch,
        "best_val_accuracy": best_val_acc,
        "best_val_macro_f1": best_val_f1,
        "hyperparameters": hyperparameters
    }

    with open(
        os.path.join(
            trial_dir,
            "summary.json"
        ),
        "w"
    ) as f:

        json.dump(
            summary,
            f,
            indent=4
        )

    print("\n" + "-" * 70)
    print(f"TRIAL {trial.number} FINISHED")
    print(f"Best Epoch: {best_epoch}")
    print(
        f"Best Validation Accuracy: "
        f"{best_val_acc * 100:.2f}%"
    )
    print(
        f"Best Validation Macro F1: "
        f"{best_val_f1:.4f}"
    )
    print("-" * 70)

    # Optuna maximizes this value
    return best_val_acc

# ============================================================
# 8. CREATE OPTUNA STUDY
# ============================================================

study = optuna.create_study(
    direction="maximize",
    study_name="ResNet18_Emotion_Classification"
)

# ============================================================
# 9. RUN OPTUNA
# ============================================================

N_TRIALS = 10

print("\n" + "=" * 70)
print("STARTING RESNET18 OPTUNA HYPERPARAMETER SEARCH")
print("=" * 70)

print("Number of trials:", N_TRIALS)

study.optimize(
    objective,
    n_trials=N_TRIALS
)

# ============================================================
# 10. PRINT BEST RESULT
# ============================================================

print("\n" + "=" * 70)
print("BEST RESNET18 HYPERPARAMETERS")
print("=" * 70)

print(
    "Best Validation Accuracy:",
    f"{study.best_value * 100:.2f}%"
)

print("\nBest Parameters:")

for parameter, value in study.best_params.items():

    print(
        f"{parameter}: {value}"
    )

# ============================================================
# 11. SAVE OPTUNA RESULTS
# ============================================================

best_result = {
    "best_validation_accuracy": study.best_value,
    "best_parameters": study.best_params,
    "best_trial_number": study.best_trial.number
}

with open(
    os.path.join(
        RESULTS_DIR,
        "best_optuna_result.json"
    ),
    "w"
) as f:

    json.dump(
        best_result,
        f,
        indent=4
    )

# ============================================================
# 12. SAVE ALL TRIAL RESULTS
# ============================================================

all_trials = []

for trial in study.trials:

    all_trials.append(
        {
            "trial": trial.number,
            "state": str(trial.state),
            "value": trial.value,
            "params": trial.params
        }
    )

with open(
    os.path.join(
        RESULTS_DIR,
        "all_optuna_trials.json"
    ),
    "w"
) as f:

    json.dump(
        all_trials,
        f,
        indent=4
    )

print("\nAll Optuna results saved to:")



[I 2026-09-26 00:54:16,674] A new study created in memory with name: ResNet18_Emotion_Classification


Using device: cuda
Results will be saved to:
/content/drive/MyDrive/Colab Notebooks/ResNet18_Optuna_Results

STARTING RESNET18 OPTUNA HYPERPARAMETER SEARCH
Number of trials: 10

STARTING OPTUNA TRIAL 0
Hyperparameters:
Learning rate : 0.00012613667067271666
Batch size    : 32
Weight decay  : 0.0002945836303656251
Dropout       : 0.45535451529705184
Freeze model  : False
Epoch [1/15] Train Loss: 0.7321 | Train Acc: 70.13% | Val Loss: 0.9725 | Val Acc: 59.15% | Val F1: 0.5871
  ✓ Best model saved! (59.15%)
Epoch [2/15] Train Loss: 0.1140 | Train Acc: 97.04% | Val Loss: 1.3061 | Val Acc: 59.86% | Val F1: 0.5891
  ✓ Best model saved! (59.86%)
Epoch [3/15] Train Loss: 0.0480 | Train Acc: 99.11% | Val Loss: 1.3008 | Val Acc: 60.56% | Val F1: 0.5964
  ✓ Best model saved! (60.56%)
Epoch [4/15] Train Loss: 0.0241 | Train Acc: 99.49% | Val Loss: 1.4984 | Val Acc: 54.23% | Val F1: 0.5367
  No improvement (1/3)
Epoch [5/15] Train Loss: 0.0144 | Train Acc: 99.87% | Val Loss: 1.5946 | Val Acc: 60.56

[I 2026-09-26 00:56:02,879] Trial 0 finished with value: 0.6056338028169014 and parameters: {'lr': 0.00012613667067271666, 'batch_size': 32, 'weight_decay': 0.0002945836303656251, 'dropout': 0.45535451529705184, 'freeze_backbone': False}. Best is trial 0 with value: 0.6056338028169014.


Epoch [6/15] Train Loss: 0.0077 | Train Acc: 99.90% | Val Loss: 1.5937 | Val Acc: 60.56% | Val F1: 0.5992
  No improvement (3/3)
Early stopping triggered!

----------------------------------------------------------------------
TRIAL 0 FINISHED
Best Epoch: 3
Best Validation Accuracy: 60.56%
Best Validation Macro F1: 0.5964
----------------------------------------------------------------------

STARTING OPTUNA TRIAL 1
Hyperparameters:
Learning rate : 0.000316926960287222
Batch size    : 32
Weight decay  : 8.461746989714442e-05
Dropout       : 0.34144919263380413
Freeze model  : False
Epoch [1/15] Train Loss: 0.6632 | Train Acc: 73.78% | Val Loss: 1.6108 | Val Acc: 56.34% | Val F1: 0.5702
  ✓ Best model saved! (56.34%)
Epoch [2/15] Train Loss: 0.1365 | Train Acc: 95.59% | Val Loss: 1.8201 | Val Acc: 56.34% | Val F1: 0.5403
  No improvement (1/3)
Epoch [3/15] Train Loss: 0.1307 | Train Acc: 95.77% | Val Loss: 1.7668 | Val Acc: 61.27% | Val F1: 0.5966
  ✓ Best model saved! (61.27%)
Epoch [4

[I 2026-09-26 00:58:31,500] Trial 1 finished with value: 0.6197183098591549 and parameters: {'lr': 0.000316926960287222, 'batch_size': 32, 'weight_decay': 8.461746989714442e-05, 'dropout': 0.34144919263380413, 'freeze_backbone': False}. Best is trial 1 with value: 0.6197183098591549.


...

Epoch [5/15] Train Loss: 0.0319 | Train Acc: 99.90% | Val Loss: 1.1148 | Val Acc: 61.27% | Val F1: 0.6055
  No improvement (3/3)
Early stopping triggered!

----------------------------------------------------------------------
TRIAL 7 FINISHED
Best Epoch: 2
Best Validation Accuracy: 63.38%
Best Validation Macro F1: 0.6297
----------------------------------------------------------------------

STARTING OPTUNA TRIAL 8
Hyperparameters:
Learning rate : 0.00021218536748887162
Batch size    : 64
Weight decay  : 1.1226436274509233e-05
Dropout       : 0.3865701249602317
Freeze model  : False
Epoch [1/15] Train Loss: 0.7068 | Train Acc: 70.99% | Val Loss: 1.1029 | Val Acc: 57.75% | Val F1: 0.5720
  ✓ Best model saved! (57.75%)
Epoch [2/15] Train Loss: 0.0968 | Train Acc: 97.68% | Val Loss: 1.1854 | Val Acc: 64.79% | Val F1: 0.6457
  ✓ Best model saved! (64.79%)
Epoch [3/15] Train Loss: 0.0285 | Train Acc: 99.29% | Val Loss: 1.4209 | Val Acc: 59.86% | Val F1: 0.5909
  No improvement (1/3)
Epoch 

[I 2026-09-26 01:13:49,615] Trial 8 finished with value: 0.647887323943662 and parameters: {'lr': 0.00021218536748887162, 'batch_size': 64, 'weight_decay': 1.1226436274509233e-05, 'dropout': 0.3865701249602317, 'freeze_backbone': False}. Best is trial 5 with value: 0.6549295774647887.


Epoch [5/15] Train Loss: 0.0086 | Train Acc: 99.85% | Val Loss: 1.2851 | Val Acc: 62.68% | Val F1: 0.6174
  No improvement (3/3)
Early stopping triggered!

----------------------------------------------------------------------
TRIAL 8 FINISHED
Best Epoch: 2
Best Validation Accuracy: 64.79%
Best Validation Macro F1: 0.6457
----------------------------------------------------------------------

STARTING OPTUNA TRIAL 9
Hyperparameters:
Learning rate : 0.00023015774518260257
Batch size    : 64
Weight decay  : 1.8452108364776197e-05
Dropout       : 0.3207367195310086
Freeze model  : False
Epoch [1/15] Train Loss: 0.6427 | Train Acc: 74.52% | Val Loss: 1.1691 | Val Acc: 59.15% | Val F1: 0.5957
  ✓ Best model saved! (59.15%)
Epoch [2/15] Train Loss: 0.0787 | Train Acc: 98.16% | Val Loss: 1.3076 | Val Acc: 62.68% | Val F1: 0.6252
  ✓ Best model saved! (62.68%)
Epoch [3/15] Train Loss: 0.0198 | Train Acc: 99.69% | Val Loss: 1.4013 | Val Acc: 60.56% | Val F1: 0.5974
  No improvement (1/3)
Epoch 

[I 2026-09-26 01:15:17,798] Trial 9 finished with value: 0.6267605633802817 and parameters: {'lr': 0.00023015774518260257, 'batch_size': 64, 'weight_decay': 1.8452108364776197e-05, 'dropout': 0.3207367195310086, 'freeze_backbone': False}. Best is trial 5 with value: 0.6549295774647887.


Epoch [5/15] Train Loss: 0.0233 | Train Acc: 99.52% | Val Loss: 1.4312 | Val Acc: 60.56% | Val F1: 0.6004
  No improvement (3/3)
Early stopping triggered!

----------------------------------------------------------------------
TRIAL 9 FINISHED
Best Epoch: 2
Best Validation Accuracy: 62.68%
Best Validation Macro F1: 0.6252
----------------------------------------------------------------------

BEST RESNET18 HYPERPARAMETERS
Best Validation Accuracy: 65.49%

Best Parameters:
lr: 6.497409727582049e-05
batch_size: 16
weight_decay: 0.00012168816738551532
dropout: 0.45185613635781285
freeze_backbone: False

All Optuna results saved to:


In [12]:
# ============================================================
# 13. DOWNLOAD BEST OPTUNA MODEL
# ============================================================

from google.colab import files

best_trial_number = study.best_trial.number

best_model_path = os.path.join(
    RESULTS_DIR,
    f"trial_{best_trial_number}",
    "checkpoints",
    "best_model.pth"
)

print("Best Trial:", best_trial_number)
print("Best Model Path:", best_model_path)

if os.path.exists(best_model_path):

    files.download(best_model_path)

else:

    print("Best model file was not found.")

Best Trial: 5
Best Model Path: /content/drive/MyDrive/Colab Notebooks/ResNet18_Optuna_Results/trial_5/checkpoints/best_model.pth


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
files.download(best_model_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
# ============================================================
# 14. RESNET18 - FINAL CLASSIFICATION REPORT
# ============================================================

import os
import torch
import torch.nn as nn
from torchvision import models
from torch.utils.data import DataLoader
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score
)

# ============================================================
# 1. SETTINGS
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class_names = [
    "Angry",
    "Fear",
    "Happy",
    "Sad"
]

best_trial_number = study.best_trial.number

best_model_path = os.path.join(
    RESULTS_DIR,
    f"trial_{best_trial_number}",
    "checkpoints",
    "best_model.pth"
)

print("=" * 70)
print("RESNET18 FINAL TEST EVALUATION")
print("=" * 70)

print("Best Trial:", best_trial_number)
print("Model:", best_model_path)


# ============================================================
# 2. LOAD BEST RESNET18 MODEL
# ============================================================

best_params = study.best_trial.params

model = models.resnet18(weights=None)

model.fc = nn.Sequential(
    nn.Dropout(best_params["dropout"]),
    nn.Linear(
        model.fc.in_features,
        NUM_CLASSES
    )
)

checkpoint = torch.load(
    best_model_path,
    map_location=device
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model = model.to(device)
model.eval()

print("✅ Best ResNet18 model loaded")


# ============================================================
# 3. TEST DATALOADER
# ============================================================

# IMPORTANT:
# test_dataset must be your UNSEEN NewArts2 test dataset.
# Do NOT use the validation dataset here.

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Test images:", len(test_dataset))


# ============================================================
# 4. GET PREDICTIONS
# ============================================================

all_labels = []
all_predictions = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)

        outputs = model(images)

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_labels.extend(
            labels.numpy()
        )


# ============================================================
# 5. ACCURACY + MACRO F1
# ============================================================

test_accuracy = accuracy_score(
    all_labels,
    all_predictions
)

test_macro_f1 = f1_score(
    all_labels,
    all_predictions,
    average="macro"
)


# ============================================================
# 6. CLASSIFICATION REPORT
# ============================================================

report = classification_report(
    all_labels,
    all_predictions,
    target_names=class_names,
    digits=4
)

print("\n" + "=" * 70)
print("FINAL TEST RESULTS")
print("=" * 70)

print(f"Test Accuracy : {test_accuracy * 100:.2f}%")
print(f"Macro F1      : {test_macro_f1:.4f}")

print("\nClassification Report:")
print(report)


# ============================================================
# 7. CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    all_labels,
    all_predictions
)

print("\n" + "=" * 70)
print("CONFUSION MATRIX")
print("=" * 70)

print(cm)


# ============================================================
# 8. SAVE EVERYTHING
# ============================================================

report_path = os.path.join(
    RESULTS_DIR,
    f"trial_{best_trial_number}",
    "final_test_classification_report.txt"
)

with open(report_path, "w") as f:

    f.write("RESNET18 FINAL TEST CLASSIFICATION REPORT\n")
    f.write("=" * 70 + "\n\n")

    f.write(f"Best Trial: {best_trial_number}\n")
    f.write(f"Test Accuracy: {test_accuracy * 100:.2f}%\n")
    f.write(f"Macro F1: {test_macro_f1:.4f}\n\n")

    f.write("Best Hyperparameters:\n")

    for key, value in best_params.items():
        f.write(f"{key}: {value}\n")

    f.write("\nClassification Report:\n")
    f.write(report)

    f.write("\nConfusion Matrix:\n")
    f.write(str(cm))


print("\n✅ Report saved to:")
print(report_path)

RESNET18 FINAL TEST EVALUATION
Best Trial: 5
Model: /content/drive/MyDrive/Colab Notebooks/ResNet18_Optuna_Results/trial_5/checkpoints/best_model.pth
✅ Best ResNet18 model loaded
Test images: 407

FINAL TEST RESULTS
Test Accuracy : 68.30%
Macro F1      : 0.6803

Classification Report:
              precision    recall  f1-score   support

       Angry     0.6707    0.5500    0.6044       100
        Fear     0.5366    0.6535    0.5893       101
       Happy     0.8661    0.9700    0.9151       100
         Sad     0.6667    0.5660    0.6122       106

    accuracy                         0.6830       407
   macro avg     0.6850    0.6849    0.6803       407
weighted avg     0.6844    0.6830    0.6790       407


CONFUSION MATRIX
[[55 30  4 11]
 [13 66  4 18]
 [ 1  1 97  1]
 [13 26  7 60]]

✅ Report saved to:
/content/drive/MyDrive/Colab Notebooks/ResNet18_Optuna_Results/trial_5/final_test_classification_report.txt


In [19]:
# ============================================================
# DOWNLOAD FINAL RESNET18 MODEL
# ============================================================

from google.colab import files
import os

# Exact path of the best model
best_model_path = (
    "/content/drive/MyDrive/Colab Notebooks/"
    "ResNet18_Optuna_Results/trial_5/checkpoints/best_model.pth"
)

print("Model path:")
print(best_model_path)

# Check that the file exists
if os.path.isfile(best_model_path):

    file_size = os.path.getsize(best_model_path) / (1024 * 1024)

    print("✅ Model found!")
    print(f"File size: {file_size:.2f} MB")
    print("Starting download...")

    files.download(best_model_path)

else:

    print("❌ Model not found!")
    print("Please check the path above.")

Model path:
/content/drive/MyDrive/Colab Notebooks/ResNet18_Optuna_Results/trial_5/checkpoints/best_model.pth
✅ Model found!
File size: 128.06 MB
Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>